In [3]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import os
import warnings
warnings.filterwarnings('ignore')

# Base path
base_path = r'C:\Users\Amsha\Desktop\EduMindAI\data\raw'
processed_path = r'C:\Users\Amsha\Desktop\EduMindAI\data\processed'
os.makedirs(processed_path, exist_ok=True)

print(" Libraries imported!")

 Libraries imported!


In [4]:
# Load all datasets
df_performance = pd.read_csv(os.path.join(base_path, 'student-por.csv'), sep=',')
df_mental = pd.read_csv(os.path.join(base_path, 'Student Mental health.csv'))
df_dropout = pd.read_csv(os.path.join(base_path, 'dataset.csv'))
df_behavior = pd.read_csv(os.path.join(base_path, 'xAPI-Edu-Data.csv'))

print("✅ All datasets loaded!")
print(f"📊 Performance: {df_performance.shape}")
print(f"😰 Mental Health: {df_mental.shape}")
print(f"🚨 Dropout: {df_dropout.shape}")
print(f"📱 Behavior: {df_behavior.shape}")

✅ All datasets loaded!
📊 Performance: (649, 33)
😰 Mental Health: (101, 11)
🚨 Dropout: (4424, 35)
📱 Behavior: (480, 17)


In [6]:
#PERFORMANCE DATASET PREPROCESSING
print("📊 Processing Performance Dataset..")
#step-1 copy dataset
df_perf = df_performance.copy()
#step-2 encode yes/no columns to 1/0
yes_no_columns=['schoolsup','famsup','paid','activities','nursery','higher','internet','romantic']
for col in yes_no_columns:
    df_perf[col]=df_perf[col].map({'yes':1,'no':0})
#step-3 encode other categorical columns
df_perf['sex']=df_perf['sex'].map({'F':0,'M':1})
df_perf['address']=df_perf['address'].map({'U':1,'R':0})
df_perf['famsize']=df_perf['famsize'].map({'GT3':1,'LE3':0})
df_perf['Pstatus']=df_perf['Pstatus'].map({'T':1,'A':0})
#step-4 encode remaining columns
le = LabelEncoder()
df_perf['Mjob'] = le.fit_transform(df_perf['Mjob'])
df_perf['Fjob'] = le.fit_transform(df_perf['Fjob'])
df_perf['reason'] = le.fit_transform(df_perf['reason'])
df_perf['guardian'] = le.fit_transform(df_perf['guardian'])
df_perf['school'] = le.fit_transform(df_perf['school'])
#step-5 create risk label
#G3 < 10 = Fail (1 = at risk), G3 >= 10 = Pass(0 = safe)
df_perf['at_risk'] = (df_perf['G3'] < 10).astype(int)

print(f"✅ Performance dataset processed!")
print(f"Shape: {df_perf.shape}")
print(f"\nAt Risk students: {df_perf['at_risk'].sum()}")
print(f"Safe students: {(df_perf['at_risk'] == 0).sum()}")
print(f"\nSample:\n{df_perf.head(3)}")

📊 Processing Performance Dataset..
✅ Performance dataset processed!
Shape: (649, 34)

At Risk students: 100
Safe students: 549

Sample:
   school  sex  age  address  famsize  Pstatus  Medu  Fedu  Mjob  Fjob  ...  \
0       0    0   18        1        1        0     4     4     0     4  ...   
1       0    0   17        1        1        1     1     1     0     2  ...   
2       0    0   15        1        0        1     1     1     0     2  ...   

   freetime  goout  Dalc  Walc  health  absences  G1  G2  G3  at_risk  
0         3      4     1     1       3         4   0  11  11        0  
1         3      3     1     1       3         2   9  11  11        0  
2         3      2     2     3       3         6  12  13  12        0  

[3 rows x 34 columns]


In [8]:
#Mental health dataset preprocessing
print(" Processing Mental Health Dataset...")

df_ment = df_mental.copy()

df_ment.columns = ['timestamp', 'gender', 'age', 'course', 
                   'year', 'cgpa', 'marital', 'depression', 
                   'anxiety', 'panic_attack', 'treatment']

yes_no_cols = ['depression', 'anxiety', 'panic_attack', 'treatment']
for col in yes_no_cols:
    df_ment[col] = df_ment[col].map({'Yes': 1, 'No': 0})

df_ment['gender'] = df_ment['gender'].map({'Female': 0, 'Male': 1})
df_ment['marital'] = df_ment['marital'].map({'No': 0, 'Yes': 1})

le = LabelEncoder()
df_ment['year'] = le.fit_transform(df_ment['year'])
df_ment['cgpa'] = le.fit_transform(df_ment['cgpa'])
df_ment['course'] = le.fit_transform(df_ment['course'])

df_ment = df_ment.drop('timestamp', axis=1)
df_ment['age'] = df_ment['age'].fillna(df_ment['age'].median())

df_ment['mental_risk'] = (
    df_ment['depression'] + 
    df_ment['anxiety'] + 
    df_ment['panic_attack']
)

print(f" Mental health dataset processed!")
print(f"Shape: {df_ment.shape}")
print(f"\nMental Risk Distribution:")
print(df_ment['mental_risk'].value_counts())

 Processing Mental Health Dataset...
 Mental health dataset processed!
Shape: (101, 11)

Mental Risk Distribution:
mental_risk
0    37
1    36
2    18
3    10
Name: count, dtype: int64


In [11]:
#Dropout dataset preprocessing
print("Processing dropout dataset...")
#step-1 copy dataset
df_drop = df_dropout.copy()
#step-2 encode target column
df_drop['Target'] = df_drop['Target'].map({
    'Graduate': 0,
    'Enrolled': 1,
    'Dropout': 2
})
#step-3 create binary dropout column
#1 = Dropout, 0= Not Dropout
df_drop['is_dropout'] = (df_drop['Target'] == 2).astype(int)
#step-4 check all columns are numeric
print(f" Dropout dataset processed!")
print(f"Shape: {df_drop.shape}")
print(f"\nDropout Distribution:")
print(df_drop['is_dropout'].value_counts())
print(f"\nDropout %: {df_drop['is_dropout'].mean()*100:.1f}%")
print(f"\nSample:\n{df_drop.head(3)}")

Processing dropout dataset...
 Dropout dataset processed!
Shape: (4424, 36)

Dropout Distribution:
is_dropout
0    3003
1    1421
Name: count, dtype: int64

Dropout %: 32.1%

Sample:
   Marital status  Application mode  Application order  Course  \
0               1                 8                  5       2   
1               1                 6                  1      11   
2               1                 1                  5       5   

   Daytime/evening attendance  Previous qualification  Nacionality  \
0                           1                       1            1   
1                           1                       1            1   
2                           1                       1            1   

   Mother's qualification  Father's qualification  Mother's occupation  ...  \
0                      13                      10                    6  ...   
1                       1                       3                    4  ...   
2                      22         

In [13]:
#Behavior dataset preprocessing
print(" Processing Behavior Dataset...")
#step-1 copy dataset
df_beh = df_behavior.copy()
#step-2 encode target class
df_beh['Class'] = df_beh['Class'].map({
    'L': 0,  # Low
    'M': 1,  # Medium
    'H': 2   # High
})
#step-3 encode categorical columns
df_beh['gender'] = df_beh['gender'].map({'M': 1, 'F': 0})
df_beh['Semester'] = df_beh['Semester'].map({'F': 0, 'S': 1})
df_beh['Relation'] = df_beh['Relation'].map({'Father': 1, 'Mum': 0})
df_beh['ParentAnsweringSurvey'] = df_beh['ParentAnsweringSurvey'].map({'Yes': 1, 'No': 0})
df_beh['ParentschoolSatisfaction'] = df_beh['ParentschoolSatisfaction'].map({'Good': 1, 'Bad': 0})
df_beh['StudentAbsenceDays'] = df_beh['StudentAbsenceDays'].map({'Under-7': 0, 'Above-7': 1})
#step-4 encode remaining columns
le = LabelEncoder()
df_beh['NationalITy'] = le.fit_transform(df_beh['NationalITy'])
df_beh['PlaceofBirth'] = le.fit_transform(df_beh['PlaceofBirth'])
df_beh['StageID'] = le.fit_transform(df_beh['StageID'])
df_beh['GradeID'] = le.fit_transform(df_beh['GradeID'])
df_beh['SectionID'] = le.fit_transform(df_beh['SectionID'])
df_beh['Topic'] = le.fit_transform(df_beh['Topic'])

print(f" Behavior dataset processed!")
print(f"Shape: {df_beh.shape}")
print(f"\nClass Distribution:")
print(df_beh['Class'].value_counts())
print(f"\nSample:\n{df_beh.head(3)}")


 Processing Behavior Dataset...
 Behavior dataset processed!
Shape: (480, 17)

Class Distribution:
Class
1    211
2    142
0    127
Name: count, dtype: int64

Sample:
   gender  NationalITy  PlaceofBirth  StageID  GradeID  SectionID  Topic  \
0       1            4             4        2        1          0      7   
1       1            4             4        2        1          0      7   
2       1            4             4        2        1          0      7   

   Semester  Relation  raisedhands  VisITedResources  AnnouncementsView  \
0         0         1           15                16                  2   
1         0         1           20                20                  3   
2         0         1           10                 7                  0   

   Discussion  ParentAnsweringSurvey  ParentschoolSatisfaction  \
0          20                      1                         1   
1          25                      1                         1   
2          30                

In [15]:
df_perf.to_csv(os.path.join(processed_path, 'performance_processed.csv'), index=False)
df_ment.to_csv(os.path.join(processed_path, 'mental_processed.csv'), index=False)
df_drop.to_csv(os.path.join(processed_path, 'dropout_processed.csv'), index=False)
df_beh.to_csv(os.path.join(processed_path, 'behavior_processed.csv'), index=False)

print(" All processed datasets saved!")
print(f"\nSaved to: {processed_path}")
print("\nFiles saved:")
print("→ performance_processed.csv")
print("→ mental_processed.csv")
print("→ dropout_processed.csv")
print("→ behavior_processed.csv")

 All processed datasets saved!

Saved to: C:\Users\Amsha\Desktop\EduMindAI\data\processed

Files saved:
→ performance_processed.csv
→ mental_processed.csv
→ dropout_processed.csv
→ behavior_processed.csv
